In [0]:
import json

ruta_landing = '/Volumes/tiktok_data_eng/landing/archivos'

#obtener todos los archivos de landing
archivos = dbutils.fs.ls(ruta_landing)

#Guardar solo los json
archivos_json = [
    archivo for archivo in archivos
    if archivo.path.endswith('.json')
]

#Se saca el archivo mas nuevo
archivo_ultimo = max(archivos_json, key=lambda archivo:archivo.modificationTime)

print(f'archivo a procesar {archivo_ultimo.path}')

# Remover el dbfs para poder abrir con el open()
file_path = archivo_ultimo.path.replace('dbfs:', '')

#abrir archivo json
with open(file_path, 'r', encoding='utf-8') as archivo:
    datos = json.load(archivo)


In [0]:
from pyspark.sql.types import StructType, StructField, StringType



# 1. LISTA PARA GUARDAR LOS REGISTROS DESANIDADOS

filas = []

# 2. RECORRER CADA VIDEO


for video in datos:

    # Diccionarios anidados
    stats = video.get("stats") or {}
    author = video.get("author") or {}
    author_stats = author.get("stats") or {}
    author_links = author.get("links") or {}
    music = video.get("music") or {}

    # Crear un registro plano
    fila = {

        # INFORMACIÓN DEL VIDEO
        "search_type": str(video.get("searchType")),
        "search_hashtag": str(video.get("hashtag")),
        "video_id": str(video.get("id")),
        "description": str(video.get("desc")),
        "created_at": str(video.get("createdAt")),
        "create_time": str(video.get("createTime")),
        "video_url": str(video.get("url")),
        "region": str(video.get("region")),
        "duration": str(video.get("duration")),
        "video_width": str(video.get("width")),
        "video_height": str(video.get("height")),
        "ratio": str(video.get("ratio")),
        "definition": str(video.get("definition")),
        "bitrate": str(video.get("bitrate")),
        "cover": str(video.get("cover")),

        "is_ad": str(video.get("isAd")),
        "is_photo": str(video.get("isPhoto")),
        "is_paid_content": str(video.get("isPaidContent")),
        "description_language": str(video.get("descLanguage")),

        "duet_control": str(video.get("duetControl")),
        "stitch_control": str(video.get("stitchControl")),
        "pinned": str(video.get("pinned")),

        # ESTADÍSTICAS DEL VIDEO
        "plays": str(stats.get("plays")),
        "likes": str(stats.get("likes")),
        "comments": str(stats.get("comments")),
        "shares": str(stats.get("shares")),
        "saves": str(stats.get("saves")),
        "reposts": str(stats.get("reposts")),

        # INFORMACIÓN DEL AUTOR
        "author_id": str(author.get("id")),
        "author_sec_uid": str(author.get("secUid")),
        "author_unique_id": str(author.get("uniqueId")),
        "author_nickname": str(author.get("nickname")),
        "author_verified": str(author.get("verified")),
        "author_verify_reason": str(author.get("verifyReason")),
        "author_signature": str(author.get("signature")),
        "author_verification_type": str(author.get("verificationType")),
        "author_region": str(author.get("region")),

        # ESTADÍSTICAS DEL AUTOR
        "author_followers": str(author_stats.get("followers")),
        "author_following": str(author_stats.get("following")),
        "author_hearts": str(author_stats.get("hearts")),
        "author_videos": str(author_stats.get("videos")),
        "author_favoriting": str(author_stats.get("favoriting")),

        # REDES DEL AUTOR
        "author_instagram": str(author_links.get("instagram")),
        "author_youtube": str(author_links.get("youtube")),

        # INFORMACIÓN DE LA MÚSICA
        "music_id": str(music.get("id")),
        "music_title": str(music.get("title")),
        "music_author": str(music.get("author")),
        "music_is_original": str(music.get("isOriginal")),
        "music_duration": str(music.get("duration")),

        # LISTAS
        "hashtags": str(video.get("hashtags")),
        "mentions": str(video.get("mentions"))
    }

    filas.append(fila)



#Crear el esquema todas las columnas string
schema = StructType([
    StructField(col, StringType(), True)
    for col in filas[0].keys()
])


#Crear dataframe
df_nuevos = spark.createDataFrame(
    filas,
    schema = schema
)

df_nuevos = df_nuevos.withColumn('fecha_ingesta',current_timestamp())
#Creamos la vista temporal para los datos nuevos
df_nuevos.createOrReplaceTempView('archivos_nuevos')

print('Creada exitosamente la vista') 

In [0]:
%sql
-- agregamos con merge nuevos registros a tiktok_bronze

merge into tiktok_data_eng.bronze.tiktok_bronze target
using archivos_nuevos source
    on target.video_id = source.video_id

when not matched then
  insert *    

In [0]:
%sql
select count(*) as luego_total_datos
from tiktok_data_eng.bronze.tiktok_bronze